# Outlier detection algorithms

This notebook is dedicated to the implementation and testing of algorithms detecting outliers in a dataset.

## One-Class SVM

Let $\mathcal{X}$ be the space where data lives, and $d=\text{dim}(\mathcal{X})$. Data points are assumed to be the realization of a probability measure noted $\mathbb{P}$. We introduce the quantile function for a real-valued function $\lambda$ on measurable subsets $\mathcal{C}$ of $\mathcal{X}$: $U(\alpha)=\inf\limits_{C \in \mathcal{C}} \{\lambda (C)\backslash \mathbb{P}(C) \geq \alpha \}$, where $\alpha \in ]0, 1]$. We note $C(\alpha)$ the set that reaches the infimum for a level $\alpha$. $\lambda$ is usually the Lebesgue measure, in which case, it identifies to the generalized inverse distribution function. [1] describes an algorithm which finds regions close to $C(\alpha)$. A first idea could be to use the set separating hyperplanes $(H_{w, \rho})_{w \in \mathbb{R}^d, \rho \in \mathbb{R}}$ of general form: $H_{w, \rho}=\{x \backslash \, (w, x)_{\mathbb{R}^d} \geq \rho\}$, to find regions close to $C(\alpha)$.

For more generality, we should employ kernel machinery to map the data space $\mathcal{X}$ to the feature space $\mathcal{H}$ via a chosen feature map $\Phi$. Separating hyperplanes are then built in the feature space: $C_{w, \rho}=\{x \in \mathcal{X} \backslash \, (\Phi(x), w)_{\mathcal{H}}\geq \rho\}$; the major interest being that they translates into regions with nonlinear borders back in data space.

Take a dataset $x_1, \cdots, x_n \in \mathcal{X}$.

The objective is to present an algorithm that identifies a function $f$ taking the value $+1$ in 'small' regions that capture most of the data points and $-1$ elsewhere. The decision rule is to declare a point $x$ as an outlier if $f(x)=-1$. $f$ is chosen to be parameterized as $f(x)=\text{sgn}((w,x)-\rho)$, and the optimization problem to find optimal parameters $(w, \rho)$ is:


\begin{align*}
\min\limits_{w\in\mathcal{H}, \rho \in \mathbb{R}} & \frac12 ||w||_2^2 + \frac1{\nu n} \sum\limits_i \xi_i - \rho \\
\text{s.t} &\,\, (w, \Phi(x_i))\geq \rho-\xi_i \, \forall i \in \{1, \cdots ,n\},\\
& \xi_i \geq 0 \, \forall i \in \{1, \cdots ,n\}.
\end{align*}

with $\nu \in ]0, 1]$ a positive parameter  tuned by the operator. If $\nu$ tends towards $0$, it becomes the hard margin problem $\forall i, \xi_i=0$, i.e., no outlier is allowed in the training dataset. It always remains feasible though, since $\rho$ is not lower-bounded. More precisely, $\nu$ is a upper bound on the number of outliers and a lower bound on the number of support vector.

After introducing the Lagrangian, one deduces the dual problem:
\begin{align*}
\min\limits_{\alpha \in \mathbb{R}^n} & \frac12 \sum\limits_{i,j=1}^n \alpha_i \alpha_j k(x_i, x_j) \\
\text{s.t} &\,\, 0\leq \alpha_i \leq \frac{1}{\nu n} \, \forall i \in \{1, \cdots ,n\},\\
& \sum\limits_{i=1}^n \alpha_i=1.
\end{align*}

Because the problem is convex, $(P) \iff (D)$.

After solving this problem, hyperplane parameters are $w=\sum\limits_{i=1}^n \alpha_i \Phi(x_i)$, and $\rho=(w, \Phi(x_i))=\sum\limits_{j=1}^n \alpha_j k(x_i, x_j)$ for any index $i$ such that $\alpha_i \in ]0, \frac1{\nu n}[$. We noted $k(x_i, x_j)=(\Phi(x_i), \Phi(x_j))_{\mathcal{H}}$.

### Optimization algorithm

To solve this QP problem, one could use an off-the-shell solver, with time complexity scaling as $\mathcal{O}(n^3)$. Instead, [1] proposes an alternate version of *Sequential Minimal Optimization* (SMO) to take advantage of the simplicity of the constraints and better scale with size of dataset $n$.

At initialization, a fraction $\nu$ of index randomly drawn are set to $\frac1{\nu l}$, the remaining $1-\nu$ index are set to $0$. By the complementary condition, it means $\xi_i \neq 0$, data point with index $i$ is an outlier.

You have to chose a pair of index $(i_1, i_2)$, and update iteratively $\alpha_{i_1}, \alpha_{i_2}$.

Formally, $\alpha_{i1}, \alpha_{i2}$ are updated by:
$$
\alpha_{i_2}^{(t+1)}=\frac{\Delta (K_{i_1,i_1} - K_{i_2,i_2}) + C_1 - C_2}{K_{i_1, i_1}+K_{i_2, i_2} - 2K_{i_1, i_2}},
$$

$$
\alpha_{i_1}^{(t+1)}=\Delta - \alpha_{i_2}^{(t+1)},
$$

where $\Delta=1-\sum\limits_{i \neq \{i_1, i_2\}}\alpha_i$, $K_{i,  j}=K(x_{i}, x_{j})$, $C_1=\sum\limits_{\substack{j=1,\\ j\neq i_1}}^l\alpha_i K_{i,j}$, $C_2=\sum\limits_{\substack{j=1,\\ j\neq i_2}}^l\alpha_i K_{i,j}.$

Also, define $O_i=K_{1i}\alpha_1 + K_{2i}\alpha_2 + C_i$.


[1] [Estimating the Support of a High-Dimensional Distribution](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/tr-99-87.pdf)

In [ ]:
import numpy as np

class One_class_SVM:
    
    def __init__(self, nu: float):
        assert nu>0, "nu should be positive"
        self.nu=nu
    
    def init(self, ln: int):
        """
        Initialize Lagrange multipliers (alpha_i)_i
        ln: number of samples.
        """
        alpha=np.zeros(ln)
        index=np.random.shuffle(np.linspace(1, ln, ln, dtype=int))[:np.floor(self.nu*ln).astype(int)]
        alpha[index]=1/(ln*self.nu)
        alpha[index[0]]+=1-np.sum(alpha)
        return alpha
    def fit(self, X: np.ndarray):
        ln=len(X)
        alpha=self.init(ln)
        mask=(0<alpha) and (alpha<1/(ln*self.nu))

## ECOD